# Graduation Prediction — Step-by-Step Analysis

This notebook walks through the full pipeline from the paper:
> *Comparative Analysis of Machine Learning Algorithms for Predicting On-Time Graduation*

**Steps:**
1. Load & inspect dataset
2. Preprocessing (encoding, imputation, SMOTE)
3. Train 4 classifiers
4. Evaluate with 5 metrics
5. Feature importance analysis
6. ROC curves

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

import warnings; warnings.filterwarnings('ignore')
SEED = 42
print('Libraries loaded. SEED =', SEED)

## 1. Load Dataset

In [ ]:
DATA_PATH = '../data/student_data.csv'
df = pd.read_csv(DATA_PATH, sep=None, engine='python')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Check target column
target_col = [c for c in df.columns if c.lower() in ('target','status')][0]
print('Target column:', target_col)
print(df[target_col].value_counts())

## 2. Preprocessing

In [ ]:
df2 = df.copy()
df2[target_col] = (df2[target_col].str.strip() == 'Graduate').astype(int)
print('Class distribution (binary):')
print(df2[target_col].value_counts())
print(f'\nGraduate: {df2[target_col].mean()*100:.1f}%')

X = df2.drop(columns=[target_col])
y = df2[target_col]

for col in X.select_dtypes('object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'\nTrain: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Train class dist: Graduate={y_train.sum()} ({y_train.mean()*100:.1f}%), '
      f'Non-Graduate={(~y_train.astype(bool)).sum()} ({(1-y_train.mean())*100:.1f}%)')

In [ ]:
smote = SMOTE(random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f'After SMOTE: {len(X_train_sm)} training samples')
print(f'Graduate: {y_train_sm.sum()} | Non-Graduate: {(~y_train_sm.astype(bool)).sum()}')

## 3. Train & Evaluate

In [ ]:
models = {
    'Random Forest':      RandomForestClassifier(n_estimators=100, random_state=SEED),
    'XGBoost':            XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                                        random_state=SEED, eval_metric='logloss', verbosity=0),
    'SVM':                SVC(kernel='rbf', C=1.0, probability=True, random_state=SEED),
    'Logistic Regression':LogisticRegression(solver='lbfgs', max_iter=1000, random_state=SEED),
}

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train_sm)
X_te_sc = scaler.transform(X_test)

results = []
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

for name, model in models.items():
    Xtr = X_tr_sc if name == 'SVM' else X_train_sm
    Xte = X_te_sc if name == 'SVM' else X_test
    
    model.fit(Xtr, y_train_sm)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:, 1]
    
    results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_proba),
        '_pred':     y_pred,
        '_proba':    y_proba,
        '_model':    model,
    })

df_res = pd.DataFrame(results)[['Model','Accuracy','Precision','Recall','F1','ROC-AUC']]
for c in df_res.columns[1:]:
    df_res[c] = df_res[c].map('{:.4f}'.format)
df_res

## 4. Confusion Matrices

In [ ]:
primary = [r for r in results if r['Model'] in ('Random Forest','XGBoost','SVM')]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Confusion Matrices — RF vs XGBoost vs SVM', fontweight='bold')
for ax, res in zip(axes, primary):
    cm = confusion_matrix(y_test, res['_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Graduate','Graduate'],
                yticklabels=['Non-Graduate','Graduate'], cbar=False)
    ax.set_title(res['Model'], fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()

## 5. Feature Importance

In [ ]:
rf_res  = next(r for r in results if r['Model'] == 'Random Forest')
xgb_res = next(r for r in results if r['Model'] == 'XGBoost')
feat_names = X.columns.tolist()

rf_imp  = pd.Series(rf_res['_model'].feature_importances_, index=feat_names).nlargest(10).sort_values()
xgb_imp = pd.Series(xgb_res['_model'].feature_importances_, index=feat_names).nlargest(10).sort_values()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance — RF vs XGBoost', fontweight='bold')
ax1.barh(rf_imp.index, rf_imp.values, color='#3266ad'); ax1.set_title('Random Forest — Top 10')
ax2.barh(xgb_imp.index, xgb_imp.values, color='#e67e22'); ax2.set_title('XGBoost — Top 10')
plt.tight_layout(); plt.show()

## 6. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#3266ad','#e67e22','#27ae60']
for res, color in zip(primary, colors):
    fpr, tpr, _ = roc_curve(y_test, res['_proba'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{res['Model']} (AUC={float(df_res.loc[df_res['Model']==res['Model'],'ROC-AUC'].values[0]):.4f})")
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Random Classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()